<a id='table_of_contents'></a>

0. [Import libraries](#imports)
1. [Load data](#load_data)
2. [Initial Cleaning](#initial_cleaning)
3. [Price and Quantity Cleaning](#price_and_quantity_cleaning)
4. [Coffee Origin Cleaning](#coffee_origin_cleaning)
5. [Export processed data](#export_data)
6. [External processing](#external_processing)


# 0. Import libraries <a id='imports'></a>
[Back to top](#table_of_contents)

In [ ]:
%reload_ext autoreload
%autoreload 2

import json
import re
from datetime import datetime
from pathlib import Path

import matplotlib as mpl
import numpy as np
import pandas as pd
import pycountry
from unidecode import unidecode

from coffee.config import DATA_DIR

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
mpl.rcParams["figure.dpi"] = 300

# 1. Import raw data <a id='load_data'></a>
[Back to top](#table_of_contents)

In [ ]:
# Set up directories
FILE_IN: str = "reviews.csv"

# Cleaning parameters
MAX_AGTRON: int = 100  # agtron readings above this are website typos
CPI_BASELINE_DATE: str = "2024-06-01"  # reference month for inflation adjustment
RANDOM_STATE: int = 0  # deterministic sampling in the display cells below

# Load data
df_in: pd.DataFrame = pd.read_csv(DATA_DIR / "raw" / FILE_IN)
df_in.info()

# 2. Initial Cleaning <a id='initial_cleaning'></a>
[Back to top](#table_of_contents)

Basic data checks and cleaning: renaming and combining columns, dropping
unnecessary columns, setting datatypes, and string cleaning.

In [ ]:
# Cleanup column names
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names: strip, lowercase, snake_case, drop dots."""
    df = df.copy()
    df.columns = (
        df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(".", "")
    )
    return df


df = df_in.pipe(clean_columns)
df.info()

In [ ]:
def tweak_df(df: pd.DataFrame) -> pd.DataFrame:
    """Initial data cleaning"""
    numeric_cols = [
        "agtron_external",
        "agtron_ground",
        "acidity",
        "rating",
        "aroma",
        "body",
        "flavor",
        "aftertaste",
    ]
    return (
        df.assign(
            review_date=lambda df_: pd.to_datetime(df_["review_date"], format="%B %Y"),
            # Combine acidity and acidity/structure into one column; they are the
            # same field but the name used in reviews changed at one point.
            acidity=lambda df_: df_["acidity"].fillna(df_["acidity/structure"]),
            # Split agtron into external- and ground-bean readings.
            agtron_external=lambda df_: pd.to_numeric(
                df_["agtron"].str.split("/", expand=True)[0].str.strip(),
                errors="coerce",
            ),
            agtron_ground=lambda df_: pd.to_numeric(
                df_["agtron"].str.split("/", expand=True)[1].str.strip(),
                errors="coerce",
            ),
            # Espresso if the title mentions it or a with-milk score is present.
            is_espresso=lambda df_: (
                df_["title"].str.contains("espresso", case=False, na=False)
                | df_["with_milk"].notna()
            ),
        )
        .replace(["", "NR", "N/A", "na"], np.nan)
        # Drop agtron typos (> MAX_AGTRON); keep rows with missing agtron.
        .loc[
            lambda df_: (
                ~(
                    (df_["agtron_external"] > MAX_AGTRON)
                    | (df_["agtron_ground"] > MAX_AGTRON)
                )
            ),
            :,
        ]
        # Run str.strip on every string cell.
        .map(lambda x: x.strip() if isinstance(x, str) else x)
        # errors="ignore": the scraped schema drifts with a page's vintage, so
        # a column being absent is normal rather than a failure.
        .drop(columns=["acidity/structure", "agtron"], errors="ignore")
        # Coerce score columns to numeric; a few rows carry qualitative acidity
        # values (e.g. "Very Low") that become NaN.
        .assign(
            **{
                col: lambda df_, col=col: pd.to_numeric(df_[col], errors="coerce")
                for col in numeric_cols
            }
        )
    )


df = df.pipe(tweak_df)
df.info()

# 3. Price and Quantity Cleaning <a id='price_and_quantity_cleaning'></a>
[Back to top](#table_of_contents)

The `est_price` column contains information on coffe roast price, quantity, and currency. We split this column to separate price and quantity information. Splitting on "/" creates one column with the price and currency and another with the quantity and unit of measure. Quantities need to be standardized to a single representation for each unit. We also filter out all products that came in units of cans, boxes, capusles, pods, etc. We will ignore extra-processed coffee and only concern ourselves with coffee sold in bags or bulk, whether ground or whole.

### Cleaning quantities

In [ ]:
# Defining list of quantity terms to drop from the dataset
drop_terms: list[str] = [
    "can",
    "box",
    "capsules",
    "K-",
    "cups",
    "bags",
    "concentrate",
    "discs",
    "bottle",
    "pods",
    "ml",
    "pouch",
    "packet|tin",
    "instant",
    "sachet",
    "vue",
    "single-serve",
    "fluid",
    "capsultes",
]

# Build a regex string to match any of the drop terms
drop_terms_string: str = "|".join(drop_terms)


def price_quantity_split(df: pd.DataFrame) -> pd.DataFrame:
    """Split the est_price column into price and quantity columns"""
    price_quantity = (
        df
        # Split est_price into columns for price and quantity
        .est_price.str.split("/", n=1, expand=True)
        # Remove any commas from the price and quantity columns
        .replace(",", "", regex=True)
        .rename(columns={0: "price", 1: "quantity"})
        .assign(
            # Cleanup quantity
            quantity=lambda df_: (
                df_["quantity"]
                .str.lower()
                .str.strip()
                # Remove parentheses and anything inside them
                .str.replace(r"\(.*?\)", "", regex=True)
                # Remove anything after a semicolon. This is usually a note,
                # or deal price.
                .str.replace(r";.*", "", regex=True)
                # Standardize units
                .str.replace(r".g$", " grams", regex=True)
                .str.replace(r"\sg$", "grams", regex=True)
                .str.replace(r"\bgram$", "grams", regex=True)
                .str.replace(r"pound$", "1 pounds", regex=True)
                .str.replace(r"oz|onces|ouncues|ounce$|ounces\*", "ounces", regex=True)
                .str.replace("kilogram", "kilograms")
                .str.replace("kg", "kilograms")
                # Remove "online" from any quantity
                .str.replace("online", "")
                .str.strip()
            ),
            price=lambda df_: df_["price"].str.replace("..", "."),
        )
        .dropna()
        # Remove rows where coffee is sold in a can, box, pouch, packet, or tin
        .loc[
            lambda df_: ~df_["quantity"].str.contains(drop_terms_string, case=False),
            :,
        ]
        # Split quantity into value and unit, and split price into value and currency
        .assign(
            # Extract number value from quantity (supports decimals, e.g. "12.5 oz")
            quantity_value=lambda df_: (
                df_["quantity"].str.extract(r"(\d+(?:\.\d+)?)").astype(float)
            ),
            # Extract the unit from quantity column
            quantity_unit=lambda df_: (
                df_["quantity"]
                .str.replace(r"(\d+)", "", regex=True)
                .replace(r"\.", "", regex=True)
                .str.strip()
                .mask(lambda s: s == "g", "grams")
                .mask(lambda s: s == "kilo", "kilograms")
                .str.strip()
            ),
            # Extract price value from price column
            price_value=lambda df_: (
                df_["price"].str.extract(r"(\d+\.\d+|\d+)").astype(float)
            ),
            # Extract currency from price column
            price_currency=lambda df_: (
                df_["price"]
                .str.replace(",", "")
                .str.replace(r"(\d+\.\d+|\d+)", "", regex=True)
                .str.strip()
            ),
        )
        # Drop the original price and quantity columns
        .drop(columns=["price", "quantity"])
        # Remove rows where quantity_unit contains
        .loc[lambda df_: ~df_["quantity_unit"].str.contains(r"\(", regex=True), :]
    )
    print(f"Shape of original DataFrame: {df.shape}")
    print(f"Shape of price_quantity: {price_quantity.shape}")

    # Merge the price_quantity DataFrame with the original DataFrame
    return df.merge(price_quantity, how="left", left_index=True, right_index=True)


df = df.pipe(price_quantity_split)

df.quantity_unit.value_counts()

In [ ]:
def convert_to_lbs(df: pd.DataFrame) -> pd.DataFrame:
    """Create a new column with the quantity in pounds"""
    to_lbs_conversion: dict[str, float] = {
        "ounces": 1 / 16,
        "pounds": 1,
        "kilograms": 2.20462,
        "grams": 0.00220462,
    }

    df["quantity_in_lbs"] = np.round(
        df["quantity_value"] * df["quantity_unit"].map(to_lbs_conversion), 2
    )
    return df


df = df.pipe(convert_to_lbs)

df.groupby("quantity_unit")[
    ["est_price", "quantity_value", "quantity_unit", "quantity_in_lbs"]
].sample(1, random_state=RANDOM_STATE)

### Cleaning Prices and Currencies
Here we normalize the currency column to contain a standard set of ISO 4217 currency codes. This will help us with fetching exchange rates from an API later on.

In [ ]:
# Map currency symbols / aliases to ISO 4217 codes, applied after stripping the
# "$" sign. Exact whole-value matches avoid the fragility of substring replaces.
CURRENCY_MAP: dict[str, str] = {
    "": "USD",
    "US": "USD",
    "PRICE:": "USD",
    "#": "GBP",
    "£": "GBP",
    "POUND": "GBP",
    "¥": "JPY",
    "€": "EUR",
    "E": "EUR",
    "EUROS": "EUR",
    "PESOS": "MXN",
    "RMB": "CNY",
    "RM": "MYR",
    "NT": "TWD",
    "NTD": "TWD",
    "HK": "HKD",
}


def clean_currency(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize the currency column to ISO 4217 codes."""
    codes = (
        df["price_currency"]
        .str.upper()
        .str.replace("$", "", regex=False)
        .str.strip()
        .replace(CURRENCY_MAP)
    )
    return df.assign(price_currency=codes)


df = df.pipe(clean_currency)


# Check that currencies make sense from original est_price column
df.loc[:, ["est_price", "price_currency", "price_value"]].groupby(
    "price_currency"
).sample(3, replace=True, random_state=RANDOM_STATE)

In [ ]:
df.price_currency.value_counts()

#### Converting prices to 2024 USD
Using historical exchange rates we will convert all prices to USD. We then adjust prices from historical USD to 2024 USD using the BLS consumer price index.

In [ ]:
with open(DATA_DIR / "external/openex_exchange_rates.json") as f:
    currency_codes: dict[str, dict[str, float]] = dict(json.load(f))

# Tidy the nested {date: {currency: rate}} mapping into a lookup table so prices
# can be converted with a vectorized merge instead of a row-wise apply.
exchange_rates = (
    pd.DataFrame(currency_codes)
    .T.rename_axis("review_date")
    .reset_index()
    .melt(id_vars="review_date", var_name="price_currency", value_name="rate")
    .assign(review_date=lambda d: pd.to_datetime(d["review_date"]))
)


def convert_currency(df: pd.DataFrame) -> pd.DataFrame:
    """Convert prices to USD using historical rates for the review month."""
    merged = df.merge(exchange_rates, on=["review_date", "price_currency"], how="left")
    merged["price_usd"] = (merged["price_value"] / merged["rate"]).round(2)
    return merged.drop(columns="rate")


df = df.pipe(convert_currency)


df.groupby("price_currency")[
    [
        "price_usd",
        "price_value",
        "price_currency",
    ]
].sample(1, random_state=RANDOM_STATE)

In [ ]:
def load_cpi_dataframe(file_path: Path) -> pd.DataFrame:
    """Loads and transforms the CPI data."""
    try:
        cpi: pd.DataFrame = pd.read_csv(file_path)
    except FileNotFoundError as exc:
        raise FileNotFoundError(
            "CPI file is not found in the specified directory."
        ) from exc

    cpi.columns = cpi.columns.str.strip().str.lower().str.replace(" ", "_")
    return (
        cpi.drop(columns=["half1", "half2"])
        .melt(id_vars="year", var_name="month", value_name="cpi")
        .assign(
            month=lambda df_: df_["month"].apply(
                lambda x: datetime.strptime(x, "%b").month
            ),
            date=lambda df_: pd.to_datetime(df_[["year", "month"]].assign(day=1)),
        )
        .drop(columns=["year", "month"])
    )


def create_cpi_adjusted_price(
    df: pd.DataFrame, file_path: Path, date: str = CPI_BASELINE_DATE
) -> pd.DataFrame:
    """Adjust historical prices to baseline-month dollars using CPI data."""
    cpi: pd.DataFrame = load_cpi_dataframe(file_path)
    cpi_baseline: float = cpi.loc[cpi["date"] == date, "cpi"].values[0]

    merged = df.merge(cpi, how="left", left_on="review_date", right_on="date")
    # Keep the raw USD price where CPI is unavailable (e.g. the current month).
    merged["price_usd_adj"] = np.where(
        merged["cpi"].isna(),
        merged["price_usd"],
        (merged["price_usd"] * cpi_baseline / merged["cpi"]).round(2),
    )
    return merged


cpi_path: Path = DATA_DIR / "external" / "consumer_price_index.csv"

df = df.pipe(create_cpi_adjusted_price, file_path=cpi_path)

df.groupby("price_currency")[
    [
        "price_value",
        "price_currency",
        "price_usd",
        "review_date",
        "price_usd_adj",
    ]
].sample(1, random_state=RANDOM_STATE)

In [ ]:
# Plot the price difference between the adjusted and historical prices over time

(
    df.assign(
        price_diff=lambda df_: (
            (df_["price_usd_adj"] - df_["price_usd"]) / df_["price_usd_adj"] * 100
        )
    ).sort_values("review_date")
).plot(
    x="review_date",
    y="price_diff",
    title="% Price difference between adjusted and historical prices",
)

### Create a column for price/lb using adjusted price 

In [ ]:
# Create a new column for price per pound
def price_per_lbs(df: pd.DataFrame) -> pd.DataFrame:
    df["price_usd_adj_per_lb"] = np.round(
        df["price_usd_adj"] / df["quantity_in_lbs"], 2
    )
    return df


df = df.pipe(price_per_lbs)

df.head()

# 4. Coffee Origin Cleaning <a id='coffee_origin_cleaning'></a>
[Back to top](#table_of_contents)

In [ ]:
def tweak_countries(countries: set[str]) -> set[str]:
    # Work on a copy so the caller's set isn't mutated.
    countries = set(countries)
    remove: list[str] = [
        "american samoa",
        "united states minor outlying islands",
        "south sudan",
        "south georgia and the south sandwich islands",
        "british indian ocean territory",
        "congo, the democratic republic of the",
        "taiwan, province of china",
        "guinea",
    ]

    # discard() ignores names that aren't present (set.remove would raise).
    for r in remove:
        countries.discard(r)
    for c in list(countries):
        c_new: str = c.split(",")[0]
        countries.remove(c)
        countries.add(c_new)

    countries.add("taiwan")
    return countries


countries: set[str] = tweak_countries(
    set(unidecode(c.name.lower()) for c in pycountry.countries)
)

# Precompiled alternation for fast, vectorized origin matching. Longer names are
# listed first so multi-word countries win over their substrings.
country_pattern = re.compile(
    r"\b(" + "|".join(sorted(map(re.escape, countries), key=len, reverse=True)) + r")\b"
)

In [ ]:
def clean_origin(df: pd.DataFrame) -> pd.DataFrame:
    """Extract origin countries from the (lowercased) coffee_origin text.

    Falls back to the original text when no country is matched, so unresolved
    origins can be reconciled manually downstream.
    """
    origin = df["coffee_origin"].str.lower()

    def match(text: str) -> str:
        if pd.isna(text) or text == "":
            return ""
        found = country_pattern.findall(text)
        return ";".join(sorted(set(found))) if found else text

    return df.assign(coffee_origin=origin, origin_country=origin.apply(match))


df = df.pipe(clean_origin)

In [ ]:
# Country aliases the site uses that pycountry does not match on its own.
COUNTRY_ALIASES: dict[str, str] = {
    "south korea": "korea, republic of",
    "north korea": "korea, democratic people's republic of",
    "england": "united kingdom",
    "scotland": "united kingdom",
    "wales": "united kingdom",
    "russia": "russian federation",
    "czech republic": "czechia",
    "slovak republic": "slovakia",
    "the netherlands": "netherlands",
    "holland": "netherlands",
    "vietnam": "viet nam",
    "british colombia": "canada",  # misspelling of the province
    "british columbia": "canada",
}


def clean_roaster_location(df: pd.DataFrame) -> pd.DataFrame:
    """Split roaster_location into a country and, for the US, a state.

    CoffeeReview writes locations most-specific-first ("Portland, Oregon";
    "Taipei, Taiwan"), so the last comma-separated part is the region. A US
    address names the STATE there rather than the country, which is why the
    state list is consulted first.

    The source text is messy in ways worth handling rather than passing on:
    trailing periods ("Montana."), the site's apostrophe spelling of Hawai'i,
    a missing comma ("Scottsdale Arizona"), and common country names that are
    not pycountry's official ones ("South Korea", "England").
    """
    us_states = {
        unidecode(s.name.lower())
        for s in pycountry.subdivisions.get(country_code="US") or []
    }
    us_states |= {"district of columbia", "washington dc", "dc"}

    def normalize(text: str) -> str:
        cleaned = unidecode(str(text)).lower().replace("'", "").strip(" .")
        return re.sub(r"\s+", " ", cleaned)

    def split(text: str) -> tuple[str, str]:
        if pd.isna(text) or not str(text).strip():
            return "", ""
        region = normalize(str(text).split(",")[-1])

        # Exact state, then state-as-suffix ("scottsdale arizona"), since the
        # site sometimes omits the comma.
        if region in us_states:
            return "united states", region
        for state in us_states:
            if region.endswith(" " + state):
                return "united states", state
        if region.startswith("big island of hawaii") or region == "hawaii":
            return "united states", "hawaii"

        return COUNTRY_ALIASES.get(region, region), ""

    parts = df["roaster_location"].apply(split)
    return df.assign(
        roaster_country=[country for country, _ in parts],
        roaster_us_state=[state for _, state in parts],
    )


df = df.pipe(clean_roaster_location)
df[["roaster_location", "roaster_country", "roaster_us_state"]].head()

In [ ]:
df.origin_country.value_counts()

# 5. Export processed data <a id='export_data'></a>
[Back to top](#table_of_contents)

In [ ]:
# Sanity checks before export — fail fast if the pipeline regresses.
assert df["rating"].dropna().between(0, 100).all(), "rating outside 0-100"
assert (df["quantity_in_lbs"].dropna() > 0).all(), "non-positive quantity_in_lbs"
assert np.isfinite(df["price_usd_adj_per_lb"].dropna()).all(), "non-finite price/lb"

# Null counts per column to eyeball completeness.
df.isna().sum().sort_values(ascending=False)

In [ ]:
FILE_OUT: str = FILE_IN.replace(".csv", "_cleaned.csv")
df.to_csv(DATA_DIR / "processed" / FILE_OUT, index=False)

# 6. External Processing <a id='external_processing'></a>
[Back to top](#table_of_contents)

The data exported in the previous step will be further cleaned using [OpenRefine](https://openrefine.org/). This will include cleaning up roaster names, and cleaning up and reconciling roaster and coffee origin locations.

